## Testing Our Fine Tuned QWEN 3.5:2b Model locally

    Using ollama for the purpose

In [1]:
from ollama import generate
import json
import re
from typing import List, Dict

    Using regex to extract only useful text from whole abstract according to the clause

In [2]:
def retrieve_snippet(document: str, clause_name: str, default_padding: int = 150) -> str:
    """
    Scans the document for keywords and dynamically crops the text. 
    Applies padding for sentence-based clauses, but uses exact matches for titles.
    """
    clean_text = " ".join(document.split())
    snippet_out = ""
    
    # 1. Exact Match Heuristics (No Padding Needed)
    exact_matches = {
        "Document Name": r"(?i)^(?:exhibit.*?\s)?\s*(.*?agreement|.*?contract|.*?plan)"
    }
    
    if clause_name in exact_matches:
        match = re.search(exact_matches[clause_name], clean_text)
        if match:
            snippet_out = match.group(1).strip()
            
    # 2. Context Match Heuristics (Padding Needed)
    if not snippet_out:
        context_matches = {
        "Parties": r"(?i)(?:entered into|by and between|among)(.*?)(?:\. |WHEREAS|RECITALS)",
        "Governing Law": r"(?i)(?:governed by|construed in accordance with)(.*?)(?:\.|$)",
        "Effective Date": r"(?i)(?:effective date|entered into on|as of)(.*?)(?:\.|$)",
        # Removed the overly broad 'notice.*?terminate' trap and added 'without cause'
        "Termination For Convenience": r"(?i)(?:terminate.*?convenience|termination without cause|without cause.*?terminate)(.*?)(?:\.|$)"
    }
        
        if clause_name in context_matches:
            match = re.search(context_matches[clause_name], clean_text)
            if match:
                start = max(0, match.start() - default_padding)
                end = min(len(clean_text), match.end() + default_padding)
                snippet_out = clean_text[start:end].strip()

    # 3. Fallback if regex misses entirely
    if not snippet_out:
        fallback_keyword = clause_name.split()[0]
        match = re.search(fallback_keyword, clean_text, re.IGNORECASE)
        if match:
            start = max(0, match.start() - 50) 
            end = min(len(clean_text), match.end() + 300)
            snippet_out = clean_text[start:end].strip()
        else:
            snippet_out = clean_text[:300]
            
    # CRITICAL SECURITY FIX: Swap double quotes to single quotes to protect JSON parser
    return snippet_out.replace('"', "'")

    Passing the extracted specific clause related text to our model using ollama

In [3]:
def extract_clause(snippet: str, clause_name: str) -> Dict:
    """
    Passes the cropped snippet to the fine-tuned model using the exact 
    prompt structure and anchor schema it was trained on.
    """
    system_prompt = f"""You are a secure, local corporate legal assistant. Analyze the provided contract text to isolate information regarding the following clause: '{clause_name}'. Extract the exact details and output them in a structured JSON schema.

    EXPECTED FORMAT:
    {{"clause_identified": "{clause_name}", "extracted_value": "insert exact text here or null"}}
    
    CRITICAL RULE: The value of "clause_identified" must be exactly "{clause_name}". Do not modify it."""
    
    user_prompt = f"Contract Snippet:\n[\"{snippet}\"]\n\nTask: Extract the exact value for '{clause_name}'."
    
    raw_prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt}<|im_end|>\n<|im_start|>assistant\n"

    try:
        response = generate(
            model='legal-qwen',
            prompt=raw_prompt,
            raw=True,
            format='json',
            options={
                'temperature': 0.0, 
                'num_ctx': 2048, # Drastically reduced context window for massive speed boost
                'stop': ['<|im_end|>', '<|im_start|>']
            }
        )
        return json.loads(response['response'])
    except Exception as e:
        return {"clause_identified": clause_name, "extracted_value": f"Execution Error: {str(e)}"}

    Calling the specific chunk using the retrieve_snippet fucntion and passing extract to extract_clause function which is connected to model and collecting output, at last combining all the outputs

In [4]:
def process_document(document_text: str, target_clauses: List[str]) -> List[Dict]:
    """Manages the full ETL pipeline for a single document."""
    print(f"[*] Initializing Pipeline. Target columns: {len(target_clauses)}")
    
    results = []
    for clause in target_clauses:
        print(f"\n    -> Retrieving & Reading: {clause}...")
        
        # PATCH 1: Increase padding to 300 to ensure we don't cut off Article 13.3
        snippet = retrieve_snippet(document_text, clause, default_padding=300) 
        
        print(f"       [Extracted Snippet]: \"{snippet}\"")
        
        # Step 2: Extract structured fields using fine-tuned model weights
        extraction = extract_clause(snippet, clause)
        
        # PATCH 2: Force the JSON key deterministically to prevent LLM hallucination
        extraction["clause_identified"] = clause 
        
        results.append(extraction)
        
    return results

    Some complex legal document to evaluate model

In [5]:
complex_legal_document = """
UNITED STATES SECURITIES AND EXCHANGE COMMISSION
WASHINGTON, D.C. 20549
EXHIBIT 10.1: MATERIAL CONTRACTS

*** TEXT OMITTED PURSUANT TO CONFIDENTIAL TREATMENT REQUEST ***
[DRAFT VERSION 9.4 - EXECUTION COPY - HIGHLY CONFIDENTIAL]

MASTER DEFINITIVE MERGER AND INTELLECTUAL PROPERTY CONSOLIDATION AGREEMENT

This Master Definitive Merger and Intellectual Property Consolidation Agreement (hereinafter referred to as the "Agreement", the "Merger Pact", or the "Consolidation Agreement") is entered into by and between Aegis Synthetics PLC, a public limited company formed under the laws of England and Wales with its principal place of business at 100 Bishopsgate, London ("Acquiring Party" or "Aegis"), Quantum Dynamix LLC, a Delaware limited liability company located at 400 Silicon Way, San Jose, California ("Target Entity" or "Quantum"), and Vanguard Holdings Group, a Cayman Islands exempted company acting as the administrative guarantor ("Guarantor").

WHEREAS, the Parties previously engaged in a Non-Disclosure Agreement dated March 4, 2025, and a non-binding Letter of Intent signed on October 12, 2025;
WHEREAS, Aegis desires to acquire all outstanding intellectual property assets, software source code, and hardware schematics owned by Quantum;
WHEREAS, the Guarantor has agreed to underwrite the financial escrow required to close this transaction;
NOW, THEREFORE, in consideration of the mutual covenants contained herein, and for other good and valuable consideration, the receipt and sufficiency of which are hereby acknowledged, the Parties agree to the following terms:

Article 1: Definitions and Interpretations.
1.1 "Licensed IP" means all patents, trademarks, trade secrets, and copyrighted material listed in Schedule C.
1.2 "Closing Date" means the date on which all financial transfers have cleared the Guarantor's escrow accounts.
1.3 "Key Personnel" refers to the C-suite executives of Quantum as defined in Appendix 2.

Article 2: Term and Effectiveness.
2.1 Signatures and Execution. The Parties acknowledge that physical signatures were captured on various dates ranging from December 15, 2026, to January 4, 2027. 
2.2 Commencement. Notwithstanding the signature dates or the provisional Letter of Intent, the substantive obligations, licensing transfers, and legal enforcement of this Agreement shall become legally binding and fully operational as of the 14th day of February, 2027 (the "Effective Date").

Article 3: Intellectual Property Transfer.
3.1 Upon the Effective Date, Quantum shall irrevocably transfer all Licensed IP to Aegis.
3.2 Quantum retains a limited, non-exclusive, non-transferable, royalty-free license to use the Licensed IP solely for internal research and development for a period of 24 months.

Article 4: Financial Compensation and Escrow.
4.1 The total purchase price is valued at $450,000,000 USD.
4.2 Aegis shall deposit an initial sum of $100,000,000 USD into the Guarantor's escrow account within 5 business days of the Effective Date.

Article 5 through Article 12: [Omitted for Brevity in Public Filing - Pertains to regulatory compliance, GDPR adherence, environmental safety standards, employee severance packages, and localized tax obligations in the UK and California].

Article 13: Term and Termination.
13.1 Termination for Material Breach. If any Party commits a material breach of its obligations under Article 3 or Article 4, the non-breaching Party must provide written notice detailing the breach. The breaching Party shall have forty-five (45) days to cure the defect. If uncured, the non-breaching Party may terminate this Agreement immediately.
13.2 Force Majeure. No Party shall be liable for failure to perform due to acts of God, global pandemics, or governmental embargoes.
13.3 Termination Without Cause. Despite the strict compliance rules outlined in Section 13.1, Aegis may elect to terminate this Agreement for convenience at any time, for any operational reason, provided that they deliver written notice of such termination to Quantum no less than ninety (90) days prior to the desired termination date.
13.4 Effect of Termination. Upon termination, all unpaid financial obligations become immediately void, and Quantum must destroy all Aegis-provided hardware.

Article 14: Indemnification and Liability Caps.
14.1 Quantum agrees to indemnify Aegis against any third-party claims asserting that the Licensed IP infringes upon existing patents.
14.2 Total liability for any damages arising under this Agreement shall be strictly capped at the total purchase price paid to date.

Article 15: Governing Law and Dispute Resolution.
15.1 Arbitration Mandate. All disputes must first go through binding arbitration in Geneva, Switzerland, under the WIPO Expedited Arbitration Rules.
15.2 Substantive Law. Regardless of the arbitration venue in Switzerland, the Parties explicitly reject the application of the United Nations Convention on Contracts for the International Sale of Goods. Instead, all contractual disputes, interpretations of clauses, and enforcement actions shall be governed by and construed in accordance with the domestic laws of the State of Delaware, explicitly excluding any conflict-of-law principles that would require the application of the laws of any other jurisdiction.

IN WITNESS WHEREOF, the duly authorized representatives of the Parties have executed this Master Definitive Merger and Intellectual Property Consolidation Agreement.
"""

In [6]:
targets = [
        "Document Name",
        "Parties",
        "Governing Law",
        "Effective Date",
        "Termination For Convenience"
]

print("[*] Launching Retriever-Reader Pipeline...")
final_output = process_document(complex_legal_document, targets)
    
print("\n================ OUTPUT ================")
print(json.dumps(final_output, indent=4))
print("==========================================")

[*] Launching Retriever-Reader Pipeline...
[*] Initializing Pipeline. Target columns: 5

    -> Retrieving & Reading: Document Name...
       [Extracted Snippet]: "UNITED STATES SECURITIES AND EXCHANGE COMMISSION WASHINGTON, D.C. 20549 EXHIBIT 10.1: MATERIAL CONTRACTS *** TEXT OMITTED PURSUANT TO CONFIDENTIAL TREATMENT REQUEST *** [DRAFT VERSION 9.4 - EXECUTION COPY - HIGHLY CONFIDENTIAL] MASTER DEFINITIVE MERGER AND INTELLECTUAL PROPERTY CONSOLIDATION AGREEMENT"

    -> Retrieving & Reading: Parties...
       [Extracted Snippet]: "9.4 - EXECUTION COPY - HIGHLY CONFIDENTIAL] MASTER DEFINITIVE MERGER AND INTELLECTUAL PROPERTY CONSOLIDATION AGREEMENT This Master Definitive Merger and Intellectual Property Consolidation Agreement (hereinafter referred to as the 'Agreement', the 'Merger Pact', or the 'Consolidation Agreement') is entered into by and between Aegis Synthetics PLC, a public limited company formed under the laws of England and Wales with its principal place of business at 100 

- After multiple testing with different prompts we found that the small model trained on cleaned data can't always handle raw legal complex text, leaving out atleast one column null with is not good for real world.